In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR       = Path.cwd().parent.parent / "data" / "raw"
PROCESSED_DIR = Path.cwd().parent.parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Normal baseline anchors (from validated CFD run, t > 10s)
P_A_NORM = 76520.14
P_B_NORM = 38311.03
P_C_NORM =   189.49
V_NORM   =   2.9499
GRAD_NORM = (P_A_NORM - P_C_NORM) / 40.0   # 1908.3 Pa/m

# Rolling window (5 timesteps = 0.5s — captures short-term transient dynamics)
WINDOW = 5

### Load & combine all three datasets

In [8]:
df = pd.read_csv(RAW_DIR /"copper_tailings_pipeline_data.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Rename to match your preprocessing script's expected column names
df = df.rename(columns={
    "pressure_A": "node_a_pressure",
    "pressure_B": "node_b_pressure",
    "pressure_C": "node_c_pressure",
    "velocity_A": "velocity_a",
    "velocity_B": "velocity_b",
    "velocity_C": "velocity_c",
})

# Rename scenario to scenario_id to match preprocessing script expectations
df["scenario_id"] = df["scenario"] + "_run" + df["run_id"].astype(str)
print(f"Loaded: {len(df):,} rows | {df['label'].nunique()} classes")
print(df.groupby("label")["scenario_id"].count().rename("rows"))


Loaded: 151,416 rows | 3 classes
label
0    50472
1    50472
2    50472
Name: rows, dtype: int64


### Feature engineering

In [9]:
# Sort so rolling features compute correctly within each run
df = df.sort_values(["scenario_id", "timestep"]).reset_index(drop=True)

# ── Group-aware rolling (per run, no leakage across runs) ────────────────────
def rolling_per_run(series, run_col, window, func):
    return df.groupby(run_col)[series.name].transform(
        lambda x: getattr(x.rolling(window, min_periods=1), func)()
    )

# ── 1. Pressure differential features ────────────────────────────────────────
# Core hydraulic signatures — primary discriminators between classes
df["pressure_drop_ab"]  = df["node_a_pressure"] - df["node_b_pressure"]
df["pressure_drop_bc"]  = df["node_b_pressure"] - df["node_c_pressure"]
df["pressure_drop_ac"]  = df["node_a_pressure"] - df["node_c_pressure"]

# ── 2. Hydraulic gradient features ───────────────────────────────────────────
# Gradient should be ~1908 Pa/m for normal; deviates under fault
# x positions: A=5m, B=25m, C=45m → distances 20m, 20m, 40m
df["grad_ab"] = df["pressure_drop_ab"] / 20.0   # Pa/m
df["grad_bc"] = df["pressure_drop_bc"] / 20.0
df["grad_ac"] = df["pressure_drop_ac"] / 40.0

# ── 3. Deviation from normal baseline ────────────────────────────────────────
# Signed deviation: positive = above normal, negative = below
# Leak → pB, pC negative; Blockage → pA, pB positive
df["dev_p_a"] = df["node_a_pressure"] - P_A_NORM
df["dev_p_b"] = df["node_b_pressure"] - P_B_NORM
df["dev_p_c"] = df["node_c_pressure"] - P_C_NORM
df["dev_v_b"] = df["velocity_b"]      - V_NORM
df["dev_v_c"] = df["velocity_c"]      - V_NORM

# ── 4. Velocity differential features ────────────────────────────────────────
# Blockage: vB spikes; Leak: vB/vC drops
df["vel_drop_ab"] = df["velocity_a"] - df["velocity_b"]
df["vel_drop_bc"] = df["velocity_b"] - df["velocity_c"]
df["vel_drop_ac"] = df["velocity_a"] - df["velocity_c"]

# ── 5. Midpoint deviation features ───────────────────────────────────────────
# Expected midpoint pressure (linear interpolation of normal gradient)
# P_mid_expected = (P_A + P_C) / 2 under uniform gradient
df["expected_p_b"]               = (df["node_a_pressure"] + df["node_c_pressure"]) / 2.0
df["midpoint_pressure_deviation"] = df["node_b_pressure"] - df["expected_p_b"]
# Positive = blockage (P_B elevated above linear); Negative = leak (P_B depleted)

df["expected_v_b"]               = (df["velocity_a"] + df["velocity_c"]) / 2.0
df["midpoint_velocity_deviation"] = df["velocity_b"] - df["expected_v_b"]
# Positive = blockage throat velocity spike; Near-zero = normal; Negative = downstream leak

# ── 6. Pressure asymmetry ratio ───────────────────────────────────────────────
# Ratio of upstream to downstream drop — changes character under each fault
# Normal: ~1.0 (symmetric gradient); Blockage: >1 (upstream elevated);
# Leak between A and C: asymmetric drop
df["grad_ratio_ab_bc"] = df["grad_ab"] / (df["grad_bc"].abs() + 1e-6)

# ── 7. Velocity-pressure coupling features ───────────────────────────────────
# Bernoulli-consistent: rising velocity should drop pressure (blockage throat)
# Leaks: both velocity and pressure drop downstream simultaneously
df["vp_coupling_b"] = df["dev_v_b"] * df["dev_p_b"]   # +ve = both same direction
df["vp_coupling_c"] = df["dev_v_c"] * df["dev_p_c"]

# ── 8. Normalised pressure ratio features ────────────────────────────────────
df["p_ratio_ba"] = df["node_b_pressure"] / (df["node_a_pressure"] + 1e-6)
df["p_ratio_ca"] = df["node_c_pressure"] / (df["node_a_pressure"] + 1e-6)
df["p_ratio_cb"] = df["node_c_pressure"] / (df["node_b_pressure"] + 1e-6)

# ── 9. Rolling statistics (per run, window=5 timesteps = 0.5s) ───────────────
# Captures short-term temporal dynamics — transient decay, oscillation
rolling_targets = [
    "node_a_pressure", "node_b_pressure", "node_c_pressure",
    "velocity_b", "velocity_c",
    "pressure_drop_ab", "pressure_drop_bc",
    "midpoint_pressure_deviation", "midpoint_velocity_deviation",
]

for col in rolling_targets:
    grp = df.groupby("scenario_id")[col]
    df[f"rolling_mean_{col}"] = grp.transform(
        lambda x: x.rolling(WINDOW, min_periods=1).mean()
    )
    df[f"rolling_std_{col}"] = grp.transform(
        lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0)
    )

# ── 10. Rate of change features ──────────────────────────────────────────────
# dP/dt at each node — captures transient spike decay rate
# Important: in steady state these → 0; during transient they're large
for col in ["node_a_pressure", "node_b_pressure", "node_c_pressure",
            "velocity_b", "velocity_c"]:
    df[f"roc_{col}"] = df.groupby("scenario_id")[col].transform(
        lambda x: x.diff().fillna(0)
    )


### Final column selection & save

In [10]:
# Keep metadata columns that match your preprocessing script exactly
METADATA_COLS = ["scenario_id", "timestep", "label"]

# All engineered feature columns
RAW_FEATURES = [
    "node_a_pressure", "node_b_pressure", "node_c_pressure",
    "velocity_a", "velocity_b", "velocity_c",
]

ENGINEERED_FEATURES = [
    # Pressure differentials
    "pressure_drop_ab", "pressure_drop_bc", "pressure_drop_ac",
    # Hydraulic gradients
    "grad_ab", "grad_bc", "grad_ac",
    # Deviations from normal
    "dev_p_a", "dev_p_b", "dev_p_c", "dev_v_b", "dev_v_c",
    # Velocity differentials
    "vel_drop_ab", "vel_drop_bc", "vel_drop_ac",
    # Midpoint deviation
    "midpoint_pressure_deviation", "midpoint_velocity_deviation",
    # Ratios
    "grad_ratio_ab_bc", "p_ratio_ba", "p_ratio_ca", "p_ratio_cb",
    # Coupling
    "vp_coupling_b", "vp_coupling_c",
    # Rolling stats
    *[f"rolling_mean_{c}" for c in rolling_targets],
    *[f"rolling_std_{c}"  for c in rolling_targets],
    # Rate of change
    "roc_node_a_pressure", "roc_node_b_pressure", "roc_node_c_pressure",
    "roc_velocity_b", "roc_velocity_c",
]

ALL_COLS = METADATA_COLS + RAW_FEATURES + ENGINEERED_FEATURES
output_df = df[ALL_COLS].copy()

out_path = PROCESSED_DIR / "live_feature_dataset.csv"
output_df.to_csv(out_path, index=False, float_format="%.6f")


# Save feature names
feature_cols = [c for c in df.columns
                if c not in ["scenario_id", "timestep", "label",
                             "fault_type", "effect_factor"]]
feature_path = Path.cwd().parent.parent / "data" / "processed" / "feature_names.txt"
with open(feature_path, "w") as f:
    for feat in feature_cols:
        f.write(feat + "\n")

print(f"Feature names saved to: {feature_path}")
print(f"Total ML features: {len(feature_cols)}")

Feature names saved to: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed/feature_names.txt
Total ML features: 56


### Summary

In [11]:
n_features = len(RAW_FEATURES) + len(ENGINEERED_FEATURES)
print("=" * 60)
print(f"  Saved → {out_path}")
print(f"  Shape : {output_df.shape}")
print(f"  Raw features       : {len(RAW_FEATURES)}")
print(f"  Engineered features: {len(ENGINEERED_FEATURES)}")
print(f"  Total features     : {n_features}")
print(f"  Metadata cols      : {METADATA_COLS}")
# save the 
print("=" * 60)
print()
print("Feature groups:")
print(f"  Pressure differentials  : pressure_drop_ab/bc/ac")
print(f"  Hydraulic gradients     : grad_ab/bc/ac")
print(f"  Baseline deviations     : dev_p_a/b/c, dev_v_b/c")
print(f"  Velocity differentials  : vel_drop_ab/bc/ac")
print(f"  Midpoint deviation      : midpoint_pressure/velocity_deviation")
print(f"  Ratios                  : grad_ratio_ab_bc, p_ratio_ba/ca/cb")
print(f"  VP coupling             : vp_coupling_b/c")
print(f"  Rolling mean/std        : {len(rolling_targets)*2} cols (window={WINDOW})")
print(f"  Rate of change          : roc_* (5 cols)")
print("=" * 60)
print()
print("Label distribution:")
print(output_df["label"].value_counts().sort_index().rename("rows"))
print()
print("Column names match preprocessing script: scenario_id, timestep, label ")
print("Ready to feed into 02_preprocessing.ipynb ")


  Saved → /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed/live_feature_dataset.csv
  Shape : (151416, 54)
  Raw features       : 6
  Engineered features: 45
  Total features     : 51
  Metadata cols      : ['scenario_id', 'timestep', 'label']

Feature groups:
  Pressure differentials  : pressure_drop_ab/bc/ac
  Hydraulic gradients     : grad_ab/bc/ac
  Baseline deviations     : dev_p_a/b/c, dev_v_b/c
  Velocity differentials  : vel_drop_ab/bc/ac
  Midpoint deviation      : midpoint_pressure/velocity_deviation
  Ratios                  : grad_ratio_ab_bc, p_ratio_ba/ca/cb
  VP coupling             : vp_coupling_b/c
  Rolling mean/std        : 18 cols (window=5)
  Rate of change          : roc_* (5 cols)

Label distribution:
label
0    50472
1    50472
2    50472
Name: rows, dtype: int64

Column names match preprocessing script: scenario_id, timestep, label 
Ready to feed into 02_preprocessing.ipynb 


In [12]:
output_df.head(10)

,scenario_id,timestep,label,node_a_pressure,node_b_pressure,node_c_pressure,velocity_a,velocity_b,velocity_c,pressure_drop_ab,...,rolling_std_velocity_c,rolling_std_pressure_drop_ab,rolling_std_pressure_drop_bc,rolling_std_midpoint_pressure_deviation,rolling_std_midpoint_velocity_deviation,roc_node_a_pressure,roc_node_b_pressure,roc_node_c_pressure,roc_velocity_b,roc_velocity_c
0,blockage_25_run0,0,2,0.000000,0.000000,0.000000,2.951007,2.951007,2.951007,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,blockage_25_run0,1,2,117289.689466,57665.388553,245.993913,4.001445,3.001142,3.001098,59624.300913,...,0.035420,42160.747499,40601.643322,779.552089,0.353645,117289.689466,57665.388553,245.993913,0.050135,0.050091
2,blockage_25_run0,2,2,115089.912449,56606.885212,238.286542,4.001398,3.001385,3.001193,58483.027237,...,0.028948,34099.423846,32851.965870,623.852482,0.288687,-2199.777017,-1058.503341,-7.707371,0.000243,0.000095
3,blockage_25_run0,3,2,113035.851969,55594.219694,231.778470,4.001781,3.000838,3.001208,57441.632275,...,0.025080,29271.735377,28204.245067,533.867327,0.250116,-2054.060480,-1012.665518,-6.508072,-0.000547,0.000015
4,blockage_25_run0,4,2,111099.763026,54644.155216,224.895595,4.001291,3.001167,3.000970,56455.607810,...,0.022410,25965.720524,25020.890691,472.542925,0.223681,-1936.088943,-950.064478,-6.882875,0.000329,-0.000238
5,blockage_25_run0,5,2,109255.053833,53751.103945,218.879717,4.001341,3.000919,3.001202,55503.949888,...,0.000102,1624.604590,1538.335896,43.594480,0.000307,-1844.709193,-893.051271,-6.015878,-0.000248,0.000232
6,blockage_25_run0,6,2,107516.185519,52898.333278,212.754993,4.001543,3.001031,3.000917,54617.852241,...,0.000142,1529.349060,1454.924530,37.485924,0.000304,-1738.868314,-852.770667,-6.124724,0.000112,-0.000285
7,blockage_25_run0,7,2,105863.203393,52098.046641,207.612585,4.001579,3.001071,3.001032,53765.156752,...,0.000133,1453.861215,1372.801787,40.648607,0.000253,-1652.982126,-800.286637,-5.142408,0.000040,0.000115
8,blockage_25_run0,8,2,104319.570061,51335.614075,202.107811,4.001600,3.001063,3.001287,52983.955986,...,0.000157,1373.677777,1299.305014,37.409717,0.000166,-1543.633332,-762.432566,-5.504774,-0.000008,0.000255
9,blockage_25_run0,9,2,102846.417466,50619.739758,198.733399,4.001683,3.001046,3.001179,52226.677708,...,0.000147,1295.463968,1229.940858,32.910075,0.000087,-1473.152595,-715.874317,-3.374412,-0.000017,-0.000108
